In [0]:
%sql
USE CATALOG db_healthcare_kl;
USE SCHEMA omop;

In [0]:
condition_occurrence_df = spark.table("condition_occurrence")

condition_occurrence_df.createOrReplaceTempView("condition_occurrence_view")

In [0]:
person_df = spark.table("person_data")

person_df.createOrReplaceTempView("person_data_view")


In [0]:
display(person_df)

In [0]:
%sql
CREATE OR REPLACE TABLE db_healthcare_kl.gold.patient_cohorts AS

select person_id, 'Diabetes' AS Cohort_name 
from condition_occurrence_view
where condition_name = 'Diabetes'

union all

SELECT
    person_id,
    'HyperTension' AS cohort_name
FROM condition_occurrence
WHERE condition_name = 'HyperTension'

union all
SELECT
    person_id,
    'Age_Above_60' AS cohort_name
FROM person_data_view
WHERE floor(months_between(current_date(), Date_of_birth)/12) >= 60



In [0]:
%sql
select * from db_healthcare_kl.gold.patient_cohorts

In [0]:
encounter_df=spark.table("visit_occurrence")

encounter_df.createOrReplaceTempView("encounter_occurrence_view")


In [0]:
display(encounter_df)

In [0]:
%sql
create or replace table db_healthcare_kl.gold.Encounter_summaries

SELECT
    person_id,
    COUNT(visit_occurrence_id) AS total_visits,
    MAX(datediff(visit_end_date, visit_start_date)) AS max_duration_days,
    AVG(datediff(visit_end_date, visit_start_date)) AS avg_duration_days
FROM encounter_occurrence_view
GROUP BY person_id

In [0]:
%sql
select * from db_healthcare_kl.gold.Encounter_summaries

In [0]:
%sql

CREATE OR REPLACE TABLE db_healthcare_kl.gold.readmission_within_30days AS

WITH visits AS (
    SELECT
        person_id,
        visit_occurrence_id,
        visit_start_date,
        visit_end_date,

        LEAD(visit_start_date) OVER (
            PARTITION BY person_id
            ORDER BY visit_start_date
        ) AS next_visit_start_date

    FROM visit_occurrence
)

SELECT
    person_id,
    visit_occurrence_id,
    visit_end_date,
    next_visit_start_date,

    CASE
        WHEN datediff(next_visit_start_date, visit_end_date) <= 30
             AND next_visit_start_date IS NOT NULL
        THEN 1
        ELSE 0
    END AS readmission_30day_flag

FROM visits;


In [0]:
%sql
SELECT * FROM db_healthcare_kl.gold.readmission_within_30days;

In [0]:
df = spark.table("omop.visit_occurrence")
df.printSchema()